# 07b - FAISS, on toy data, before trusting it on real data

Notebook 08 jumps straight into building two FAISS indices over the real
complaint embeddings. This is a quick detour to actually understand the
library's API on small, hand-crafted vectors first -- where the "right
answer" is obvious by inspection -- before trusting it on 6,150-dimensional
real data.

One thing worth saying up front: at this project's scale (thousands of
complaints), brute-force cosine similarity via `sklearn.metrics.pairwise.cosine_similarity`
(what notebooks 05-07 used) is already fast enough. FAISS isn't needed here
for speed. The reason this project uses it (decided before notebook 08) is
the clean "index + `.iloc[]` metadata lookup" pattern -- no new abstraction
on top of how this project already works with pandas, not a performance
requirement.

## Setup

In [1]:
import numpy as np
import faiss
from sklearn.metrics.pairwise import cosine_similarity


## Toy vectors

Six 4-dimensional vectors, built so the "obvious" similarity structure is
visible just by looking at the numbers: A and A' point in nearly the same
direction, B and B' point in nearly the same direction, C and D are both
distinct from everything else and from each other.

In [2]:
labels = ["A", "A'", "B", "B'", "C", "D"]

vectors = np.array([
    [1.0, 0.0, 0.0, 0.0],   # A
    [0.9, 0.1, 0.0, 0.0],   # A' -- close to A
    [0.0, 1.0, 0.0, 0.0],   # B
    [0.0, 0.9, 0.1, 0.0],   # B' -- close to B
    [0.0, 0.0, 1.0, 0.0],   # C
    [0.0, 0.0, 0.0, 1.0],   # D
], dtype="float32")

for label, vec in zip(labels, vectors):
    print(f"{label}: {vec}")


A: [1. 0. 0. 0.]
A': [0.9 0.1 0.  0. ]
B: [0. 1. 0. 0.]
B': [0.  0.9 0.1 0. ]
C: [0. 0. 1. 0.]
D: [0. 0. 0. 1.]


## `IndexFlatL2` -- nearest neighbours by raw distance

The simplest FAISS index: no approximation, just exact L2 (Euclidean)
distance. `index.search(queries, k)` returns `(distances, indices)` -- for
an L2 index, FAISS returns *squared* L2 distance, and results are sorted
smallest-first (smallest distance = most similar).

In [3]:
index_l2 = faiss.IndexFlatL2(vectors.shape[1])
index_l2.add(vectors)

query = vectors[0:1]  # A, as a query -- must be 2D: shape (1, dim)
distances, indices = index_l2.search(query, k=3)

print("Query: A")
for dist, idx in zip(distances[0], indices[0]):
    print(f"  {labels[idx]:>3}  squared L2 distance = {dist:.4f}")


Query: A
    A  squared L2 distance = 0.0000
   A'  squared L2 distance = 0.0200
   B'  squared L2 distance = 1.8200


A's nearest neighbour (other than itself) is A', exactly as expected from
just looking at the numbers. So far so good -- but notice the distance
*values* themselves aren't easy to interpret as a similarity score, and
they're sensitive to vector magnitude, not just direction. That matters for
embeddings, where what we actually care about is direction (semantic
similarity), not magnitude.

## `IndexFlatIP` on raw vectors -- a gotcha

Inner product (`IndexFlatIP`) is the other basic FAISS metric, and results
are sorted largest-first (largest inner product = most similar). But raw
inner product depends on magnitude, not just direction. To show this
concretely: add a 7th vector that points in the *exact same direction* as A,
just scaled up. A magnitude-blind similarity measure should call this a
perfect match to A -- raw inner product does not.

In [4]:
vectors_with_scaled = np.vstack([vectors, vectors[0:1] * 3.0])  # "A-scaled": same direction as A, 3x the length
labels_with_scaled = labels + ["A-scaled"]

index_ip_raw = faiss.IndexFlatIP(vectors_with_scaled.shape[1])
index_ip_raw.add(vectors_with_scaled)

scores, indices = index_ip_raw.search(vectors[0:1], k=3)

print("Query: A -- raw inner product, unnormalized")
for score, idx in zip(scores[0], indices[0]):
    print(f"  {labels_with_scaled[idx]:>9}  inner product = {score:.4f}")


Query: A -- raw inner product, unnormalized
   A-scaled  inner product = 3.0000
          A  inner product = 1.0000
         A'  inner product = 0.9000


"A-scaled" outranks A' even though A-scaled points in *exactly* the same
direction as the query and A' doesn't -- raw inner product rewards magnitude,
which isn't the semantic signal we actually want from sentence embeddings.

## Normalize first -- now `IndexFlatIP` *is* cosine similarity

`faiss.normalize_L2` rescales every vector to unit length in place. Once all
vectors (and the query) have unit length, inner product is mathematically
identical to cosine similarity -- and direction-only, magnitude-blind, the
way we actually want it.

In [5]:
vectors_norm = vectors_with_scaled.copy()
faiss.normalize_L2(vectors_norm)

index_ip_norm = faiss.IndexFlatIP(vectors_norm.shape[1])
index_ip_norm.add(vectors_norm)

query_norm = vectors[0:1].copy()
faiss.normalize_L2(query_norm)

scores, indices = index_ip_norm.search(query_norm, k=3)

print("Query: A -- normalized inner product (= cosine similarity)")
for score, idx in zip(scores[0], indices[0]):
    print(f"  {labels_with_scaled[idx]:>9}  cosine similarity = {score:.4f}")


Query: A -- normalized inner product (= cosine similarity)
   A-scaled  cosine similarity = 1.0000
          A  cosine similarity = 1.0000
         A'  cosine similarity = 0.9939


Now A-scaled comes back with cosine similarity 1.0 -- a perfect match to A,
exactly as it should be, since it's the same direction. A' is close but not
identical, as expected. This is the recipe `build_faiss_index()` in
`src/retrieval.py` actually uses: normalize, then `IndexFlatIP`.

## Cross-check against sklearn

Same toy vectors, same question, via the brute-force method used in
notebooks 05-07. If FAISS and sklearn disagree here, something is wrong with
the FAISS setup -- this is the same kind of sanity check notebook 08 ran on
the real embeddings, just on data small enough to read by eye first.

In [6]:
sklearn_sims = cosine_similarity(vectors[0:1], vectors_with_scaled)[0]

print(f"{'label':>9}  {'faiss':>8}  {'sklearn':>8}")
for label in labels_with_scaled:
    i = labels_with_scaled.index(label)
    faiss_sim = float(vectors_norm[i] @ query_norm[0])
    print(f"{label:>9}  {faiss_sim:8.4f}  {sklearn_sims[i]:8.4f}")

max_diff = np.abs(np.array([vectors_norm[i] @ query_norm[0] for i in range(len(labels_with_scaled))]) - sklearn_sims).max()
print(f"\nMax difference: {max_diff:.8f}")


    label     faiss   sklearn
        A    1.0000    1.0000
       A'    0.9939    0.9939
        B    0.0000    0.0000
       B'    0.0000    0.0000
        C    0.0000    0.0000
        D    0.0000    0.0000
 A-scaled    1.0000    1.0000

Max difference: 0.00000000


## Trying it on a slice of real data

Same four lines, on 50 real narrative-complaint embeddings instead of toy
vectors, to bridge to how notebook 08 and `src/retrieval.py` actually use
this. The key habit to take away: FAISS only ever returns integer positions
-- `df.iloc[idx]` is what turns those positions back into something
readable, exactly like `get_neighbors` already did with sklearn in
`src/retrieval.py`.

In [7]:
import pickle
import pandas as pd

df_narrative = pd.read_parquet("../data/processed/narrative_complaints.parquet").iloc[:50].copy()
with open("../data/processed/narrative_embeddings.pkl", "rb") as f:
    sample_embeddings = pickle.load(f)[:50]

normalized = sample_embeddings.copy().astype("float32")
faiss.normalize_L2(normalized)

index_real = faiss.IndexFlatIP(normalized.shape[1])
index_real.add(normalized)

query_idx = 0
scores, indices = index_real.search(normalized[query_idx:query_idx + 1], k=4)

print(f"Query: row {query_idx} -- {df_narrative.iloc[query_idx]['Consumer complaint narrative'][:120]}")
print()
for score, idx in zip(scores[0], indices[0]):
    if idx == query_idx:
        continue
    print(f"  row {idx} (sim={score:.3f}): {df_narrative.iloc[idx]['Consumer complaint narrative'][:120]}")


Query: row 0 -- On or about XX/XX/2022 the company was contacted by me regarding information I believe inaccurate and/or incorrect being

  row 15 (sim=0.644): I found inaccurate and incorrect data on my credit report and this wrong information should not be on my credit report.
  row 37 (sim=0.594): To whom it may concern I have been contacted XXXX XXXX XXXX many times concerning my original contract with the company.
  row 6 (sim=0.546): Creditor was advised of personal difficulties regarding reduced employment and lack of financial ability to satisfy debt


## Takeaway

Four lines are doing all the work everywhere FAISS shows up in this project
(`build_faiss_index()` in `src/retrieval.py`, and notebook 08):

```python
faiss.normalize_L2(vectors)            # unit-length vectors -> inner product becomes cosine similarity
index = faiss.IndexFlatIP(dim)         # exact (not approximate) inner-product index
index.add(vectors)                     # vectors are stored at integer positions 0..N-1
scores, indices = index.search(query, k)   # indices map straight back via df.iloc[]
```

`IndexFlatIP`/`IndexFlatL2` are *exact* search -- no approximation, no
training step, which is why this is the right choice at this project's scale
(thousands of rows). FAISS also offers approximate index types (IVF, HNSW,
product quantization) for million-plus-vector datasets, traded off against
exact recall -- not needed here, and intentionally out of scope for this
notebook.